# UMI 파이프라인 — 촬영에서 ckpt 까지

셀을 **위에서 아래로** 실행한다. 각 셀은 진행 상황을 그대로 찍는다.

| 구간 | 누가 |
|---|---|
| 촬영 | 현석 (S22 앱 + UMI 그리퍼) |
| `gopro_slam` 실행 | 현석 WSL — 서버에 빌드 환경이 없다 (sudo·cmake·헤더 없음) |
| **그 외 전부** | 이 노트북 |

**게이트를 못 맞추면 게이트를 낮추지 않는다. 다시 찍는다.**

## 0. 설정 — 여기만 고친다

In [ ]:
import os, sys, json, subprocess, time, shutil
from pathlib import Path

HOME   = Path.home()
REPO   = HOME / "S15P21A103"                      # 계측기·게이트
BUILDER= HOME / "ubuntu_umi" / "02_umi_dataset_builder"
BUNDLE = HOME / "hyeonseok" / "umi_gpu_bundle"    # 학습기
PY_GATE= HOME / "envs/handoff312/bin/python"      # 게이트·감사기 (numpy 2.x)
PY_BLD = HOME / "envs/umi02/bin/python"           # 02 빌더 (numpy 1.26 고정)
PY_TRN = "python"                                 # 학습기 (시스템 파이썬, torch cu126)

# ── 이번 배치 ────────────────────────────────────────────────
NAME     = "onsite_0921"
RAW      = HOME / "data" / NAME / "raw"                  # rec_* 원본 (영상+IMU)
PROC     = HOME / "data" / NAME / "processed"            # SLAM 산출 (현석에게 받는다)
SLAM_TAG = HOME / "data" / NAME / "processed_mapping" / "tx_slam_tag.json"
CAM_TCP  = BUILDER / "configs" / "s22_camera_tcp.json"   # 현석 physically_validated 판 오면 교체
DATASET_CFG = REPO / "AI/configs/umi_builder/dataset_trim0.json"   # ⚠️ 빌더 기본값은 trim 3.0 — 파지를 지운다
OUT      = REPO / "out" / NAME
GPU      = "1"          # 1·2·3·4·6 만. 0·5·7·8·9 는 타인 것
EPOCHS   = 120
BATCH    = 64
RATE_HZ  = 30.0

OUT.mkdir(parents=True, exist_ok=True)

def run(cmd, cwd=None, env=None, title=None):
    """명령을 돌리며 출력을 **실시간으로** 흘린다. 반환은 종료코드."""
    if title: print(f"── {title}\n$ {' '.join(map(str, cmd))}\n", flush=True)
    e = dict(os.environ); e.update(env or {})
    p = subprocess.Popen([str(c) for c in cmd], cwd=cwd and str(cwd), env=e,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="", flush=True)
    p.wait()
    print(f"\n[종료코드 {p.returncode}]", flush=True)
    return p.returncode

def jload(p):
    return json.loads(Path(p).read_text(encoding="utf-8"))

print("설정 완료")
for k, v in [("REPO", REPO), ("BUILDER", BUILDER), ("BUNDLE", BUNDLE),
             ("RAW", RAW), ("PROC", PROC), ("SLAM_TAG", SLAM_TAG),
             ("CAM_TCP", CAM_TCP), ("DATASET_CFG", DATASET_CFG), ("OUT", OUT)]:
    print(f"  {k:9} {v}   {'있음' if Path(v).exists() else '없음'}")

## 1. 프리플라이트 — 없는 게 있으면 여기서 멈춘다

가용성 확인이 아니라 **기능 확인**이다. GPU 는 실제 행렬곱까지 시킨다.

In [ ]:
ok = tot = 0
def chk(name, cond, hint=""):
    global ok, tot
    tot += 1; ok += bool(cond)
    print(f"  [{'있음' if cond else '없음'}] {name}" + ("" if cond else f"   <- {hint}"))

chk("게이트용 python", PY_GATE.exists(), str(PY_GATE))
chk("빌더용 python",   PY_BLD.exists(),  f"python3 -m venv ~/envs/umi02 후 02 requirements 설치")
chk("02 빌더",         (BUILDER/"build_dataset.py").exists(), str(BUILDER))
chk("학습기",          (BUNDLE/"03_umi_policy_trainer/train_policy.py").exists(), str(BUNDLE))
for t in ("gate_raw_capture", "gate_slam_batch", "make_episode_plan", "audit_umi_zarr",
          "smoke_deploy_ckpt", "probe_chunk_anchor"):
    chk(f"계측기 {t}", (REPO/"AI/tools"/f"{t}.py").exists(), "git pull origin ai")
chk("원본 raw",   RAW.exists(),  str(RAW))
print(f"\n프리플라이트 {ok} / {tot}")
assert ok == tot, "위 [없음] 을 먼저 해결한다. 미판정은 통과가 아니다"

In [ ]:
import textwrap
code = textwrap.dedent("""
    import torch
    print("torch", torch.__version__, "available", torch.cuda.is_available())
    if torch.cuda.is_available():
        a = torch.randn(512, 512, device="cuda")
        print("device", torch.cuda.get_device_name(0), "cc", torch.cuda.get_device_capability(0))
        print("실연산 OK", round((a @ a).sum().item(), 3))
""")
rc = run([PY_TRN, "-c", code], env={"CUDA_VISIBLE_DEVICES": GPU}, title="GPU 실연산")
assert rc == 0, "GPU 가 안 된다. 학습 단계로 넘어가지 않는다"

## 2. 촬영 게이트 — SLAM 돌리기 **전에** 거른다

15항목. 불합격 편은 그 자리에서 다시 찍는다. SLAM 을 돌리고 나서 알면 늦다.

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/gate_raw_capture.py",
          "--raw", RAW, "--out", OUT/"capture_gate.json"], title="촬영 게이트")
d = jload(OUT/"capture_gate.json"); t = d["tally"]
print(f"\n>> 통과 {t.get('PASS',0)} · 불합격 {t.get('FAIL',0)} · 미판정 {t.get('INCOMPLETE',0)} / 전체 {d['total']}")

## 3. SLAM — 현석 WSL 구간

서버에 ORB-SLAM3 를 못 세운다 (sudo 불가 · cmake·OpenCV·Eigen·GLEW 헤더 없음).
**매핑 편 먼저, 그다음 시연 편을 같은 아틀라스로 localization** 한다.

현석이 WSL 에서 돌릴 명령 (참고용 — 이 셀은 안내만 출력한다)

In [ ]:
print(f"""[현석 WSL]
# 1) 매핑 편 -> 아틀라스
ORB_SLAM_SAVE_MAP=<prep_map>/map_atlas.osa INIT_TAG_ID=13 INIT_TAG_SIZE_M=0.16 \\
  sh 02_umi_dataset_builder/slam/run_orbslam_linux.sh <mapping_rec> <prep_map> <prep_map>/slam

# 2) tx_slam_tag
python build_dataset.py slam-tag <mapping_rec> <prep_map>

# 3) 시연 편마다 같은 아틀라스로 localization
ORB_SLAM_LOAD_MAP=<prep_map>/map_atlas.osa \\
  sh 02_umi_dataset_builder/slam/run_orbslam_linux.sh <rec> <prep_rec> <prep_rec>/slam

받을 것: processed/rec_*/  (slam/camera_trajectory.csv · trajectory_validation.json ·
         gripper_width.csv · telemetry.json)  +  processed_mapping/*/tx_slam_tag.json
여기로 두면 된다: {PROC}""")
print("\n현재 상태:", "processed 있음" if PROC.exists() else "processed 아직 없음 — 받은 뒤 아래 셀로")

## 4. SLAM 게이트

필수 10 · 권고 2 · 배치 1. **배치 항목이 핵심이다** — 모든 편이 같은 아틀라스를 써야 한다.
서로 다른 지도가 섞이면 좌표계가 섞이고, 그건 다른 어떤 지표에도 안 나타난다.

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/gate_slam_batch.py",
          "--processed", PROC, "--out", OUT/"slam_gate.json"], title="SLAM 게이트")
g = jload(OUT/"slam_gate.json"); t = g["tally"]
print(f"\n>> 통과 {t.get('PASS',0)} · 불합격 {t.get('FAIL',0)} · 미판정 {t.get('INCOMPLETE',0)} / 전체 {g['total']}")
for r in g["batch"]["rows"]:
    print("   ", r["name"], "->", r["got"])

## 5. episode plan — 게이트 통과 편만 담는다

🟢 2026-09-21 실증 — 92편을 그대로 넣었더니 한 편의
`marker missing run 17 exceeds limit` 로 **build 전체가 중단**됐다.
빌더는 불량 편을 건너뛰지 않는다. 그래서 plan 앞에 게이트가 있어야 한다.

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/make_episode_plan.py",
          "--gate", OUT/"slam_gate.json", "--raw", RAW, "--processed", PROC,
          "--slam-tag", SLAM_TAG, "--out", OUT/"plan.json"], title="plan 생성")
assert rc == 0
pl = jload(OUT/"plan.json")
print(f"\n>> plan 편 {len(pl['episodes'])}")

## 6. zarr 빌드

**빌더 `output` 인자는 이름을 그대로 쓴다. `.zarr.zip` 을 붙여주지 않는다.** 🟢
확장자를 뺀 이름을 주면 확장자 없는 파일이 생기고, 다음 셀이 `FileNotFoundError` 로 죽는다.

**`--dataset-config` 를 반드시 준다.** 🟢 2026-09-21 실증 —
핸드오프 기본 `configs/dataset.json` 은 `episode_start_trim_s: 3.0` 이라
편마다 앞 90프레임(30Hz×3초)이 잘려 **파지 닫힘이 0/10편**이 됐다.
그래도 빌드는 `status: pass` 로 끝난다. (MEASURE_trim_default_0921)

`--camera-tcp` 가 hand-eye 를 넣는 자리다. 핸드오프 기본값은
`cad_estimate_axis_mapping_unvalidated` · `physical_deployment_allowed: false` 다.
현석의 `physically_validated` 판을 받으면 **0번 셀의 `CAM_TCP` 를 그걸로 바꾼다.**

hand-eye 실측 차이 🟢 — v4(미검증) 위치 RMSE 41.60mm / 회전 4.27° ·
s22_pick_v3(물리검증) **24.39mm / 1.33°**

In [ ]:
ZARR   = OUT / f"ds_{NAME}.zarr.zip"          # 빌더는 이 이름을 **그대로** 쓴다
REPORT = OUT / f"ds_{NAME}.zarr.report.json"  # output.with_suffix(".report.json")
assert not ZARR.exists(), f"이미 있다. 빌더는 덮어쓰기를 거부한다: {ZARR}"

rc = run([PY_BLD, "build_dataset.py", "build", OUT/"plan.json", ZARR,
          "--dataset-config", DATASET_CFG, "--camera-tcp", CAM_TCP],
         cwd=BUILDER, title="zarr 빌드 (영상 디코딩 — 몇 분 걸린다)")
assert rc == 0, "빌드 실패. 로그 위쪽의 편 이름과 사유를 본다"
# 종료코드 0 만 보면 '성공했는데 파일이 없다' 가 통과한다 — 그게 없음과 괜찮음이 같은 출력이다
assert ZARR.exists(),   f"종료코드 0 인데 zarr 가 없다: {ZARR}"
assert REPORT.exists(), f"종료코드 0 인데 report 가 없다: {REPORT}"

print("\nzarr", ZARR, f"{ZARR.stat().st_size:,} B")
d = jload(REPORT)
for k in ("status","episodes","frames","native_sample_rate_hz","world_frame",
          "episode_start_trim_s","camera_tcp_status","training_input_status",
          "physical_deployment_ready"):
    print(f"  {k:28} {d.get(k)}")
# --dataset-config 가 먹었는지 되읽어 확인한다. 안 먹으면 3.0 이 조용히 들어온다
assert d.get("episode_start_trim_s") == 0.0, \
    f"trim 이 {d.get('episode_start_trim_s')} 다 — --dataset-config 가 안 먹었다"

## 7. zarr 감사 + 사전 등록 게이트

참조 🟢 현석 s22_pick_v3 — 경로/직선 1.63 · EEF 폭 0.257m · 회전 폭 0.127rad · 개구 p10–p90 1.4mm

In [ ]:
rc = run([PY_GATE, REPO/"AI/tools/audit_umi_zarr.py", "--zarr", ZARR,
          "--label", NAME, "--gate", "--rate-hz", RATE_HZ,
          "--out", OUT/"zarr_gate.json"], title="zarr 감사 + 게이트")
gg = jload(OUT/"zarr_gate.json").get("gates", {})
print(f"\n>> {gg.get('verdict')} · 통과 {gg.get('passed')} · 불합격 {gg.get('failed')} · 미판정 {gg.get('unknown')} / {gg.get('total')}")
for r in gg.get("rows", []):
    if r.get("ok") is not True:
        print("   미통과", r["name"], r["got"], "기준", r["want"])

## 8. 학습 — 120 epoch ≈ 20분

**평가 지표는 롤아웃 성공률이다. train_loss 도 val loss 도 아니다.**
val 0.00547→0.00517 인데 롤아웃 0% 그대로였던 실증이 있다 (독립 2회) 🟢

In [ ]:
# 6번 셀(빌드)을 건너뛰고 **이미 있는 zarr** 로 학습할 때 여기서 고른다.
# 커널을 재시작했으면 ZARR 이 정의돼 있지 않다 — 이 셀이 그 자리다.
ZARR = OUT / f"ds_{NAME}.zarr.zip"        # <- 쓸 파일만 바꾼다. 예: OUT / "ds_trim0.zarr.zip"

cands = sorted(p.name for p in OUT.glob("*.zarr.zip"))
assert ZARR.exists(), f"없다: {ZARR.name}\nOUT 안의 후보 {len(cands)}개: {cands}"
print("학습에 쓸 zarr:", ZARR, f"{ZARR.stat().st_size:,} B")
print("OUT 안의 후보:", cands)

In [ ]:
RUN_ID = f"{NAME}_{time.strftime('%H%M%S')}"
TRAINER = BUNDLE / "03_umi_policy_trainer" / "train_policy.py"
PROFILE = BUNDLE / "03_umi_policy_trainer" / "configs" / "policy_resnet18_gpu.yaml"
env = {"CUDA_VISIBLE_DEVICES": GPU, "UMI_ROOT": str(BUNDLE/"third_party/umi"),
       "WANDB_MODE": "disabled", "HYDRA_FULL_ERROR": "1"}

rc = run([PY_TRN, TRAINER, "check", ZARR], cwd=BUNDLE, env=env, title="dataset check")
assert rc == 0, "dataset check 실패 — 학습으로 넘어가지 않는다"

rc = run([PY_TRN, TRAINER, "train", ZARR, "--run-id", RUN_ID, "--runs-dir", OUT/"runs",
          "--epochs", EPOCHS, "--batch", BATCH, "--eval-batch", BATCH, "--profile", PROFILE],
         cwd=BUNDLE, env=env, title=f"학습 {EPOCHS} epoch")
assert rc == 0

In [ ]:
MAN = OUT/"runs"/RUN_ID/"manifest.json"
m = jload(MAN); b = m["best_checkpoint"]
p, h = b["policy"], b["hold_current_pose_and_width"]
print(f"epoch {b['epoch']} · baseline 이김 {b['beats_hold_baseline']}\n")
print(f"{'지표':34}{'정책':>12}{'hold':>12}{'배수':>8}")
for k in ("position_component_rmse_mm","position_distance_rmse_mm","position_distance_p95_mm",
          "rotation_mean_deg","width_rmse_mm"):
    print(f"{k:34}{p[k]:>12.3f}{h[k]:>12.3f}{h[k]/p[k]:>7.2f}x")
print("\n** 개루프 예측 오차다. 롤아웃 성공률이 아니다 **")
print("참조 🟢 현석 v4 3.52x · s22_pick_v3 3.12x · 우리 v10(10Hz) 2.96x")

## 9. 배포용 export → 배포 검사

🟢 2026-09-21 실증 — **학습 ckpt 를 검사기에 바로 넣으면 안 된다.**
학습 매니페스트에는 `actionSpec` · `nParams` · `sha256_export` 가 아예 없어서
검사기가 `KeyError: 'actionSpec'` 로 죽었고, 학습 ckpt 는 `state_dicts` 에
`model` 과 `ema_model` 을 둘 다 갖고 있어 "ema_model 뿐" 항목도 헛불합격했다.

배포 계약을 만드는 건 `export_deploy_ckpt.py` 다. **export -> 검사** 순서다.
`chunk_anchor: "chunk_start"` (D-AI-80) 도 여기서 매니페스트에 들어간다.

In [ ]:
CK   = OUT/"runs"/RUN_ID/"checkpoints"/"best.ckpt"
DCK  = OUT/"runs"/RUN_ID/"deploy"/f"{RUN_ID}.ckpt"
DMAN = DCK.with_suffix(".manifest.json")        # export 가 이 이름으로 쓴다

rc = run([PY_GATE, REPO/"AI/tools/export_deploy_ckpt.py",
          "--checkpoint", CK, "--out", DCK, "--note", f"{NAME} {RUN_ID}"],
         cwd=REPO, title="배포용 export (ema_model 만 남기고 계약 manifest 작성)")
assert rc == 0, "export 실패 — 검사 단계로 넘어가지 않는다"
assert DCK.exists() and DMAN.exists(), f"종료코드 0 인데 산출물이 없다: {DCK} · {DMAN}"

rc = run([PY_GATE, REPO/"AI/tools/smoke_deploy_ckpt.py",
          "--checkpoint", DCK, "--manifest", DMAN], cwd=REPO, title="ckpt 배포 검사")
print("\n배포 ckpt:", DCK)
print("다음 — 실물 투입은 RUNBOOK 6절. --dry-run -> 물체 없이 -> 본 실행 순서.")

## 한계 — 먼저 말한다

- **실물 재검증 미실행.** 앵커 수정이 원호를 없애는지 아직 모른다 (모터·그리퍼 고장)
- 게이트는 **데이터 품질**만 본다. 그 데이터로 학습한 정책이 잘 되는지는 말하지 않는다
- `T_camera_tcp` 가 `unvalidated` 면 report 의 `physical_deployment_ready` 가 false 로 남는다.
  그 상태 수치는 조건 병기 없이 인용하지 않는다
- SLAM 은 현석 WSL 구간이다. 서버 빌드는 sudo·헤더 부재로 막혀 있다